# 04 · Chatbot turístico completo

Integra recuperación FAISS, clasificador fine-tuneado, memoria de cinco turnos y un generador local mediante Ollama.

Antes de ejecutar: completar el notebook 03 y descargar un modelo local, por ejemplo `ollama pull qwen3:1.7b` (8 GB RAM) o `ollama pull qwen3:4b`.

In [1]:
import json
import sys
from pathlib import Path

RUTA_PROYECTO = Path.cwd().resolve()
if not (RUTA_PROYECTO / 'data').is_dir(): RUTA_PROYECTO = RUTA_PROYECTO.parent
sys.path.insert(0, str(RUTA_PROYECTO))

from app.config import CACHE_DIR, EMBEDDING_MODEL, MAX_TURNS, MODELS_DIR, OLLAMA_MODEL, TOP_K
from src.chatbot_engine import TourismChatbot
from src.finetuning_utils import load_classifier
from src.rag_utils import RAGStore

assert (MODELS_DIR / 'clasificador_tipo_lugar').exists(), 'Primero ejecuta el notebook 03.'
assert (CACHE_DIR / 'indice.faiss').exists(), 'Primero ejecuta el notebook 02.'


C:\Proyecto3_Chatbot_Turístico_Inteligente\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
rag = RAGStore(CACHE_DIR, EMBEDDING_MODEL)
clasificar, modelo_clasificador = load_classifier(MODELS_DIR / 'clasificador_tipo_lugar')
print(f'RAG: {rag.index.ntotal} chunks | Clasificador: {modelo_clasificador.config.num_labels} categorías')


RAG: 5018 chunks | Clasificador: 7 categorías


In [3]:
PERSONALIDAD = ('Eres TicoGuía, un asesor turístico profesional especializado en Costa Rica. Responde en español, con tono cordial y práctico, exclusivamente con base en las reseñas recuperadas. No inventes datos ni menciones citas, fuentes, chunks o detalles técnicos: la interfaz muestra las fuentes por separado. Si no hay evidencia suficiente, responde exactamente: No tengo información suficiente en las reseñas recuperadas para responder esa pregunta.')

class TicoGuia:
    def __init__(self, max_turnos=5):
        self.historial = deque(maxlen=max_turnos * 2)

    def _categoria(self, pregunta):
        salida = clasificar(pregunta[:512])[0][0]
        return modelo_clasificador.config.id2label[int(salida['label'].split('_')[-1])] if salida['label'].startswith('LABEL_') else salida['label']

    def _buscar(self, pregunta, categoria, top_k=4, umbral=0.50):
        vector = embedder.encode([pregunta], convert_to_numpy=True); faiss.normalize_L2(vector)
        scores, ids = indice.search(vector, min(40, indice.ntotal)); resultado=[]; vistos=set()
        for score, idx in zip(scores[0], ids[0]):
            if idx < 0 or score < umbral: continue
            chunk = chunks[idx]
            if categoria and chunk['tipo_lugar'] != categoria: continue
            if chunk['lugar'] in vistos: continue
            vistos.add(chunk['lugar']); resultado.append({**chunk, 'score': float(score)})
            if len(resultado) == top_k: break
        return resultado

    def responder(self, pregunta):
        categoria = self._categoria(pregunta)
        recuperados = self._buscar(pregunta, categoria)
        contexto = '\n\n'.join(f"[{i}] {c['tipo_lugar']} - {c['lugar']} ({c['calificacion']} estrellas, fuente: {c['fuente']}): {c['texto']}" for i,c in enumerate(recuperados,1))
        historial = '\n'.join(f"{r}: {m}" for r,m in self.historial)
        prompt = f"{PERSONALIDAD}\n\nHistorial:\n{historial or '(sin historial)'}\n\nReseñas recuperadas:\n{contexto or '(sin evidencia)'}\n\nPregunta: {pregunta}"
        salida = ollama.chat(model=MODELO_OLLAMA, messages=[{'role':'user','content':prompt}])['message']['content']
        self.historial.extend([('Usuario', pregunta), ('TicoGuía', salida)])
        return salida, recuperados, categoria

# La clase anterior ilustra el flujo; para la prueba final se usa el motor
# compartido con la aplicación Dash, evitando duplicar lógica.
chatbot = TourismChatbot(rag, clasificar, modelo_clasificador, OLLAMA_MODEL, MAX_TURNS, TOP_K)


## Pruebas conversacionales

Ejecutar y documentar al menos diez conversaciones finales: factual, comparativa, seguimiento y fuera de dominio.

In [4]:
conversacion = ['¿Qué hoteles tienen buena atención al cliente?', 'Dame otro del mismo tipo', '¿Y ese dónde queda?', '¿Cuánto cuesta un vuelo a Madrid?']
registro = []
for pregunta in conversacion:
    resultado = chatbot.respond(pregunta)
    fuentes = resultado['sources']
    registro.append({'pregunta': pregunta, 'respuesta': resultado['answer'], 'categoria_predicha': resultado['category'], 'fuentes': [{'lugar': c['lugar'], 'tipo_lugar': c['tipo_lugar'], 'score': c['score']} for c in fuentes]})
    print(f"Usuario: {pregunta}\nTicoGuía: {resultado['answer']}\nFuentes: {[c['lugar'] for c in fuentes]}\n")

# Esta prueba demuestra la memoria; las diez conversaciones de evaluación
# permanecen consolidadas en resultados/metricas.json.
ruta_pruebas = RUTA_PROYECTO / 'resultados' / 'conversaciones_memoria_notebook.json'
ruta_pruebas.write_text(json.dumps(registro, ensure_ascii=False, indent=2), encoding='utf-8')
assert registro[1]['categoria_predicha'] == registro[0]['categoria_predicha']
assert {f['lugar'] for f in registro[1]['fuentes']}.isdisjoint({f['lugar'] for f in registro[0]['fuentes']})
assert registro[2]['fuentes'][0]['lugar'] == registro[1]['fuentes'][0]['lugar']
print(f'Memoria validada. Conversación guardada en {ruta_pruebas}')


Usuario: ¿Qué hoteles tienen buena atención al cliente?
TicoGuía: Los hoteles con buena atención al cliente según las reseñas son:  
- **Hotel Alajuela City**: Recomendado por su atención excelente y personal amable.  
- **Hotel Marriott Hacienda Belén**: Atención super profesional, detalles de servicio impecables y restaurante excelente.  
- **Hotel Las Brumas**: Atención buena y instalaciones bien cuidadas.  
- **Hotel Courtyard de Marriott • Alajuela**: Se menciona como un lugar excelente, aunque no se detallan aspectos específicos de atención.  

¿Qué tipo de servicios ofrecen estos hoteles?
Fuentes: ['Hotel Alajuela City', 'Hotel Marriott Hacienda Belén', 'Hotel Las Brumas', 'Hotel Courtyard de Marriott • Alajuela']

Usuario: Dame otro del mismo tipo
TicoGuía: Como alternativa, Hotel Park View (5.0 estrellas). La reseña recuperada indica: Muy buen hotel agradable
Fuentes: ['Hotel Park View']

Usuario: ¿Y ese dónde queda?
TicoGuía: Sobre Hotel Park View, la reseña recuperada indica